# Midterm Notebook

Cleanup: 

Loading each CSV and both tables in inventory_and_sales.db. Raw dfs seperate from cleaned data, time stamps = eastern time, later converted to UTC

In [ ]:
from pathlib import Path
import re
import sqlite3

import pandas as pd

if (Path.cwd() / "data").is_dir():
    DATA_DIR = Path.cwd() / "data"
elif (Path.cwd().parent / "data").is_dir():
    DATA_DIR = Path.cwd().parent / "data"
else:
    raise FileNotFoundError("could not find the data folder from the notebook directory")

DB_PATH = DATA_DIR / "inventory_and_sales.db"
print(f"data folder: {DATA_DIR.resolve()}")

loading all csv data as strings, also reading sqlite tables

In [ ]:
# keep csv values as text while loading
raw_transactions = pd.read_csv(DATA_DIR / "walmart_transactions.csv", dtype="string", keep_default_na=False)
raw_products = pd.read_csv(DATA_DIR / "product_catalog.csv", dtype="string", keep_default_na=False)
raw_stores = pd.read_csv(DATA_DIR / "store_locations.csv", dtype="string", keep_default_na=False)
raw_distribution_centers = pd.read_csv(DATA_DIR / "distribution_centers.csv", dtype="string", keep_default_na=False)

# load both tables from the database
with sqlite3.connect(DB_PATH) as connection:
    raw_inventory = pd.read_sql_query("SELECT * FROM inventory_levels", connection)
    raw_daily_sales = pd.read_sql_query("SELECT * FROM daily_sales_summary", connection)

raw_tables = {
    "transactions": raw_transactions,
    "products": raw_products,
    "stores": raw_stores,
    "distribution_centers": raw_distribution_centers,
    "inventory_levels": raw_inventory,
    "daily_sales_summary": raw_daily_sales,
}

for name, frame in raw_tables.items():
    print(f"{name}: {frame.shape[0]:,} rows x {frame.shape[1]} columns")
    display(frame.head())

data cleaning: 
- normalizing null vals, store ids, capializing, spacing, etc. (formatting)
- getting rid of duplicate vals
- standardize products to 1 SKU & product name.
- time stamps localized to NY time and also convert to utc 
- converting prices w/ symbols
- "quarantine" negative transactions (seperate from valid values)

In [ ]:
MISSING_TOKENS = {"", "NULL", "N/A", "NA", "NAN", "NONE"}

# mark blank and null-like text as missing
def normalize_missing(frame):
    cleaned = frame.copy()
    for column in cleaned.select_dtypes(include=["object", "string"]).columns:
        values = cleaned[column].astype("string").str.strip()
        cleaned[column] = values.mask(values.str.upper().isin(MISSING_TOKENS), pd.NA)
    return cleaned

# put store ids in one format
def normalize_store_id(values):
    text = values.astype("string").str.strip().str.upper()
    parts = text.str.extract(r"^FL[\s_-]?(\d{3})$", expand=False)
    return text.where(parts.isna(), "FL-" + parts)

def normalize_category(values):
    text = values.astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
    key = text.str.casefold().str.replace(r"[^a-z0-9]", "", regex=True)
    aliases = {
        "breakfastfoods": "Breakfast", "breakfast": "Breakfast",
        "emergencysupplies": "Emergency", "emergency": "Emergency",
        "alcoholicbeverages": "Alcohol", "alcohol": "Alcohol",
        "entertainment": "Entertainment", "electronics": "Electronics",
        "groceries": "Grocery", "grocery": "Grocery",
        "householdessentials": "Household", "household": "Household",
        "babyproducts": "Baby", "baby": "Baby", "health": "Health",
        "snacks": "Snacks",
    }
    return key.map(aliases).fillna(text.str.title()).astype("string")

# remove symbols before converting to numbers
def to_number(values):
    text = values.astype("string").str.strip().str.replace(r"[$,~]", "", regex=True)
    return pd.to_numeric(text, errors="coerce")

def deduplicate(frame, keys, name):
    before = len(frame)
    result = frame.drop_duplicates(subset=keys, keep="first").copy()
    print(f"{name}: removed {before - len(result):,} duplicate rows by {keys}")
    return result

In [ ]:
sku_groups = {
    "PT-12": ["PT-12", "SKU#459812", "POPTART-STRAW", "PT_STRAWBERRY"],
    "WTR-24": ["WTR-24", "WATER24PK", "WTR_24PK"],
    "BATT-100": ["BATT-100", "BAT-AA4", "BATT_AA_4PK"],
    "FLASH-01": ["FLASH-01", "LED-FLASH"],
    "BEER-24": ["BEER-24", "LTBEER24"],
    "BG-MONO": ["BG-MONO", "MONOPOLY-01", "GAME-MONO"],
    "CARDS-01": ["CARDS-01", "PLAYCARD-STD"],
    "STREAM-01": ["STREAM-01", "ROKU-STD"],
    "SOUP-CAN": ["SOUP-CAN", "CNSOUP-01"],
    "BREAD-WH": ["BREAD-WH", "WHITEBREAD"],
    "PB-16": ["PB-16", "PNUTBTR-16"],
    "CNDL-EM": ["CNDL-EM", "CANDLE-EMRG"],
    "FAID-01": ["FAID-01", "FIRSTAID-KIT"],
    "CHRG-PORT": ["CHRG-PORT", "PORTCHARGER"],
    "CHIPS-VP": ["CHIPS-VP", "CHIPVARIETY"],
    "PTOWEL-6": ["PTOWEL-6", "PAPERTOWEL6"],
    "TP-12": ["TP-12", "TOILETPAPER12"],
    "DIAP-S3": ["DIAP-S3", "DIAPERS-3"],
    "BFORM-01": ["BFORM-01", "BABYFORMULA"],
    "TARP-HD": ["TARP-HD", "HDTARP-01"],
}
canonical_names = {
    "PT-12": "Strawberry Pop-Tarts", "WTR-24": "Bottled Water 24-Pack",
    "BATT-100": "AA Batteries 4-Pack", "FLASH-01": "LED Flashlight",
    "BEER-24": "Light Beer 24-Pack", "BG-MONO": "Monopoly Board Game",
    "CARDS-01": "Playing Cards", "STREAM-01": "Roku Streaming Stick",
    "SOUP-CAN": "Canned Soup", "BREAD-WH": "White Bread",
    "PB-16": "Peanut Butter", "CNDL-EM": "Emergency Candles",
    "FAID-01": "First Aid Kit", "CHRG-PORT": "Portable Phone Charger",
    "CHIPS-VP": "Chips Variety Pack", "PTOWEL-6": "Paper Towels",
    "TP-12": "Toilet Paper 12-Roll", "DIAP-S3": "Baby Diapers Size 3",
    "BFORM-01": "Infant Formula", "TARP-HD": "Heavy Duty Tarp",
}
sku_to_canonical = {
    alias.strip().upper(): canonical
    for canonical, aliases in sku_groups.items()
    for alias in aliases
}

# check catalog skus before mapping aliases
products = normalize_missing(raw_products)
products["sku"] = products["sku"].astype("string").str.strip().str.upper()
catalog_skus = set(products["sku"].dropna())
unmapped_catalog_skus = sorted(catalog_skus - set(sku_to_canonical))
print(f"catalog SKUs without an explicit alias: {unmapped_catalog_skus}")
assert not unmapped_catalog_skus, "add an explicit canonical mapping for every catalog SKU"

products["canonical_sku"] = products["sku"].map(sku_to_canonical)
products["product_name"] = products["canonical_sku"].map(canonical_names)
products["category"] = normalize_category(products["category"])
products["subcategory"] = products["subcategory"].astype("string").str.strip().str.title()
products["unit_cost"] = to_number(products["unit_cost"])
products = deduplicate(products, ["sku"], "product catalog")

sku_aliases = pd.DataFrame(
    [(alias, canonical, canonical_names[canonical])
     for canonical, aliases in sku_groups.items() for alias in aliases],
    columns=["source_sku", "canonical_sku", "canonical_product_name"],
)
display(sku_aliases.head(12))
display(products.head())

In [ ]:
# remove repeated transaction ids
transactions = normalize_missing(raw_transactions)
transactions["transaction_id"] = transactions["transaction_id"].astype("string").str.strip().str.upper()
transactions = deduplicate(transactions, ["transaction_id"], "transactions")

transactions["store_id"] = normalize_store_id(transactions["store_id"])
transactions["source_sku"] = transactions["sku"].astype("string").str.strip().str.upper()
transactions["canonical_sku"] = transactions["source_sku"].map(sku_to_canonical).fillna(transactions["source_sku"])
transactions["product_name"] = transactions["canonical_sku"].map(canonical_names).fillna(
    transactions["product_name"].astype("string").str.strip().str.replace(r"\s+", " ", regex=True).str.title()
)
transactions["category"] = normalize_category(transactions["category"])
transactions["quantity"] = to_number(transactions["quantity"])
transactions["unit_price"] = to_number(transactions["unit_price"])

# convert local sales times to utc
parsed_timestamps = pd.to_datetime(transactions["timestamp"], format="mixed", errors="coerce")
if parsed_timestamps.dt.tz is None:
    parsed_timestamps = parsed_timestamps.dt.tz_localize(
        "America/New_York", ambiguous="NaT", nonexistent="NaT"
    )
transactions["timestamp_local"] = parsed_timestamps
transactions["timestamp_utc"] = parsed_timestamps.dt.tz_convert("UTC")

# set negative quantities aside for review
negative_quantity_mask = transactions["quantity"].lt(0).fillna(False)
transactions_quarantined = transactions.loc[negative_quantity_mask].copy()
transactions_clean = transactions.loc[~negative_quantity_mask].copy()
print(f"quarantined negative-quantity transactions: {len(transactions_quarantined):,}")
print(f"unparsed or missing timestamps: {transactions_clean['timestamp_utc'].isna().sum():,}")
print(f"missing prices: {transactions_clean['unit_price'].isna().sum():,}")
display(transactions_clean.head())
display(transactions_quarantined.head())

In [ ]:
stores = normalize_missing(raw_stores)
stores["store_id"] = normalize_store_id(stores["store_id"])
stores["state"] = stores["state"].astype("string").str.strip().str.upper().replace(
    {"FLORIDA": "FL", "FLA.": "FL"}
)
stores["region"] = stores["region"].astype("string").str.strip().str.title()
stores["latitude"] = to_number(stores["latitude"])
stores["longitude"] = to_number(stores["longitude"])
stores["zip_code"] = stores["zip_code"].astype("string").str.strip().str.zfill(5)
stores = deduplicate(stores, ["store_id"], "store locations")

distribution_centers = normalize_missing(raw_distribution_centers)
for column in ["avg_daily_shipments", "surge_capacity", "distance_to_orlando_mi", "cost_per_shipment"]:
    distribution_centers[column] = to_number(distribution_centers[column])
distribution_centers = deduplicate(distribution_centers, ["dc_id"], "distribution centers")

transactions_clean = transactions_clean.merge(
    stores[["store_id", "store_name", "city", "state", "region"]],
    on="store_id", how="left", validate="many_to_one", suffixes=("", "_store"),
)
print(f"transactions with no matching store: {transactions_clean['store_name'].isna().sum():,}")
display(stores.head())
display(distribution_centers)

In [ ]:
inventory = normalize_missing(raw_inventory)
inventory["store_id"] = normalize_store_id(inventory["store_id"])
inventory["sku"] = inventory["sku"].astype("string").str.strip().str.upper()
inventory["source_sku"] = inventory["sku"]
inventory["canonical_sku"] = inventory["sku"].map(sku_to_canonical).fillna(inventory["sku"])
inventory["date"] = pd.to_datetime(inventory["date"], format="mixed", errors="coerce")
for column in ["units_on_hand", "units_on_order", "reorder_point"]:
    inventory[column] = to_number(inventory[column]).astype("Int64")
inventory = deduplicate(inventory, ["date", "store_id", "sku"], "inventory levels")
# combine sku aliases for each store and date
inventory = inventory.groupby(
    ["date", "store_id", "canonical_sku"], as_index=False, dropna=False
).agg(
    units_on_hand=("units_on_hand", "sum"),
    units_on_order=("units_on_order", "sum"),
    reorder_point=("reorder_point", "sum"),
    source_skus=("source_sku", lambda values: ", ".join(sorted(set(values.dropna())))),
)

daily_sales = normalize_missing(raw_daily_sales)
daily_sales["store_id"] = normalize_store_id(daily_sales["store_id"])
daily_sales["date"] = pd.to_datetime(daily_sales["date"], format="mixed", errors="coerce")
daily_sales["category"] = normalize_category(daily_sales["category"])
for column in ["total_units", "total_revenue"]:
    daily_sales[column] = to_number(daily_sales[column])
daily_sales = deduplicate(daily_sales, ["date", "store_id", "category"], "daily sales summary")

display(inventory.head())
display(daily_sales.head())

- missing fields are still missing
- transactions_quarantined has negative/null values, transactions_clean has the valid data
- time converted to utc

In [ ]:
# check the cleaned data
assert transactions_clean["transaction_id"].is_unique
assert transactions_clean["quantity"].dropna().ge(0).all()
assert transactions_clean["timestamp_utc"].dt.tz is not None
assert products["sku"].isin(sku_to_canonical).all()
assert stores["store_id"].is_unique

quality_summary = pd.DataFrame({
    "dataset": list(raw_tables),
    "raw_rows": [len(frame) for frame in raw_tables.values()],
    "clean_rows": [len(transactions_clean), len(products), len(stores),
                   len(distribution_centers), len(inventory), len(daily_sales)],
})
display(quality_summary)
print("canonical product count:", products["canonical_sku"].nunique())
print("transactions retained:", f"{len(transactions_clean):,}")
print("transactions quarantined:", f"{len(transactions_quarantined):,}")
print("stores without coordinates:", int(stores[["latitude", "longitude"]].isna().any(axis=1).sum()))
display(transactions_clean[["transaction_id", "timestamp_local", "timestamp_utc", "store_id",
                            "canonical_sku", "product_name", "category", "quantity", "unit_price"]].head(10))